In [ ]:
# ============================================================
# Network Vulnerability Detection using YOLOv8
# ============================================================

# Install Required Libraries
!pip install ultralytics opencv-python matplotlib numpy -q

In [ ]:
# Import Libraries

import os
import cv2
import yaml
import random
import shutil
import zipfile
import numpy as np
import matplotlib.pyplot as plt

from glob import glob
from ultralytics import YOLO

print("Libraries loaded successfully")

In [ ]:
# Dataset Paths

BASE_DIR = "dataset"

TRAIN_IMAGES = os.path.join(BASE_DIR, "train/images")
TRAIN_LABELS = os.path.join(BASE_DIR, "train/labels")

VAL_IMAGES = os.path.join(BASE_DIR, "val/images")
VAL_LABELS = os.path.join(BASE_DIR, "val/labels")

os.makedirs(TRAIN_IMAGES, exist_ok=True)
os.makedirs(TRAIN_LABELS, exist_ok=True)

os.makedirs(VAL_IMAGES, exist_ok=True)
os.makedirs(VAL_LABELS, exist_ok=True)

print("Dataset folders created")

In [ ]:
# Upload Dataset Zip

from google.colab import files

uploaded = files.upload()

In [ ]:
# Extract Dataset

zip_file = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(BASE_DIR)

print("Dataset extracted successfully")

In [ ]:
# Create data.yaml

data_yaml = {
    'path': os.path.abspath(BASE_DIR),
    'train': os.path.abspath(TRAIN_IMAGES),
    'val': os.path.abspath(VAL_IMAGES),

    'names': {
        0: 'ethernet_port',
        1: 'damaged_cable',
        2: 'unplugged_cable'
    }
}

with open(os.path.join(BASE_DIR, 'data.yaml'), 'w') as file:
    yaml.dump(data_yaml, file)

print("data.yaml created")

In [ ]:
# Visualize Sample Images

CLASS_NAMES = {
    0: 'ethernet_port',
    1: 'damaged_cable',
    2: 'unplugged_cable'
}

def show_sample(image_path, label_path):

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h, w, _ = image.shape

    with open(label_path, 'r') as file:
        labels = file.readlines()

    for label in labels:

        data = label.strip().split()

        class_id = int(data[0])

        x_center, y_center, width, height = map(float, data[1:])

        x1 = int((x_center - width / 2) * w)
        y1 = int((y_center - height / 2) * h)

        x2 = int((x_center + width / 2) * w)
        y2 = int((y_center + height / 2) * h)

        cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)

        cv2.putText(
            image,
            CLASS_NAMES[class_id],
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2
        )

    return image


sample_images = glob(os.path.join(TRAIN_IMAGES, "*.jpg"))

plt.figure(figsize=(15, 5))

for i in range(min(3, len(sample_images))):

    img_path = sample_images[i]

    label_path = os.path.join(
        TRAIN_LABELS,
        os.path.basename(img_path).replace(".jpg", ".txt")
    )

    result = show_sample(img_path, label_path)

    plt.subplot(1, 3, i + 1)
    plt.imshow(result)
    plt.axis("off")

plt.show()

In [ ]:
# Load YOLOv8 Nano Model

model = YOLO("yolov8n.pt")

print("YOLOv8 Nano model loaded")

In [ ]:
# Train Model

results = model.train(

    data=os.path.join(BASE_DIR, "data.yaml"),

    epochs=12,
    imgsz=640,
    batch=8,

    augment=True,
    pretrained=True,

    freeze=10,
    patience=5,

    name="network_vulnerability_model"
)

print("Training completed")

In [ ]:
# Load Best Model

best_model = YOLO(
    "runs/detect/network_vulnerability_model/weights/best.pt"
)

print("Best trained model loaded")

In [ ]:
# Upload Test Image

from google.colab import files

uploaded = files.upload()

test_image = list(uploaded.keys())[0]

print("Test image uploaded")

In [ ]:
# Run Prediction

results = best_model.predict(

    source=test_image,
    conf=0.25,
    save=True
)

print("Prediction completed")

In [ ]:
# Display Output Image

output_path = results[0].save_dir + "/" + os.path.basename(test_image)

image = cv2.imread(output_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 10))
plt.imshow(image)
plt.axis("off")
plt.title("Detection Result")

plt.show()

In [ ]:
# Vulnerability Analysis

warnings_map = {

    "damaged_cable": "Damaged Cable Detected",

    "unplugged_cable": "Open Port Risk"
}

print("\n===== Vulnerability Summary =====\n")

for result in results:

    boxes = result.boxes

    for i in range(len(boxes)):

        class_id = int(boxes.cls[i])

        confidence = float(boxes.conf[i])

        class_name = CLASS_NAMES[class_id]

        print(f"{class_name} detected ({confidence:.2f})")

        if class_name in warnings_map:

            print("⚠", warnings_map[class_name])

print("\n================================")

In [ ]:
# Save Final Output

final_output_path = "final_output.jpg"

shutil.copy(output_path, final_output_path)

print(f"Final output saved as: {final_output_path}")

In [ ]:
# Save Final Output

final_output_path = "final_output.jpg"

shutil.copy(output_path, final_output_path)

print(f"Final output saved as: {final_output_path}")

# Final Project Workflow

"""
Input Image
      ↓
YOLOv8 Object Detection
      ↓
Vulnerability Detection
      ↓
Annotated Output Image
"""